<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs the core libraries for backtesting and performance analysis used in this notebook.

In [ ]:
!pip install pandas quantstats backtrader

The openbb_terminal SDK requires its own dedicated installation process and may have system-level dependencies. Follow the official OpenBB installation guide separately before running this notebook.

## Imports and setup

We use datetime for date parsing, pandas for tabular data handling, openbb for fetching historical market data, quantstats for generating professional performance reports, and backtrader for running the backtest simulation itself.

In [ ]:
import datetime as dt
import backtrader as bt
import pandas as pd
import quantstats as qs
import yfinance as yf

## Load and prepare historical data

This function fetches historical price data for a given symbol using yFinance, saves it to CSV, and converts it into a Backtrader-compatible data feed.

In [ ]:
def yf_data_to_bt_data(symbol, start, end):
    df = yf.download(
        symbol, start=start, end=end,
        auto_adjust=False,
        multi_level_index=False,
        progress=False,
    )

    fn = f"{symbol.lower()}.csv"
    df.to_csv(fn)

    return bt.feeds.YahooFinanceCSVData(
        dataname=fn,
        fromdate=dt.datetime.strptime(start, '%Y-%m-%d'),
        todate=dt.datetime.strptime(end, '%Y-%m-%d')
    )

Backtrader expects data in a specific format, so we save the OpenBB output as a CSV and reload it through Backtrader's built-in Yahoo Finance CSV parser. This intermediate step bridges two libraries that don't natively share a data format. The date range is explicitly passed to both the data fetch and the feed to ensure consistency.

This helper calculates the last calendar day of any given month, which the strategy needs to determine whether we're near month-end.

In [ ]:
def last_day_of_month(any_day):
    next_month = any_day.replace(day=28) + dt.timedelta(days=4)
    return (next_month - dt.timedelta(days=next_month.day)).day

The trick here is that day 28 exists in every month, so adding 4 days always lands in the next month. Subtracting back by the resulting day number gives us the last day of the original month. This avoids importing the calendar module for a simple calculation.

## Define the monthly flows strategy

This strategy class encodes a calendar-based trading rule: go short during the first week of each month and go long during the last week. Backtrader strategies inherit from bt.Strategy and implement their logic in the next() method, which is called once per bar (each trading day).

In [ ]:
class MonthlyFlows(bt.Strategy):
    params = (
        ("end_of_month", 23),
        ("start_of_month", 7),
    )
    def __init__(self):
        self.order = None
        self.dataclose = self.datas[0].close
    def notify_order(self, order):
        self.order = None
    def next(self):
        dt_ = self.datas[0].datetime.date(0)
        dom = dt_.day
        ldm = last_day_of_month(dt_)
        if self.order:
            return
        if not self.position:
            if dom <= self.params.start_of_month:
                self.order = self.order_target_percent(target=-1)
                print(
                    f"Created SELL of {self.order.size} "
                    f"at {self.dataclose[0]} on day {dom}"
                )
            if dom >= self.params.end_of_month:
                self.order = self.order_target_percent(target=1)
                print(
                    f"Created BUY of {self.order.size} "
                    f"{self.dataclose[0]} on day {dom}"
                )
        else:
            if self.position.size > 0:
                if not self.params.end_of_month <= dom <= ldm:
                    print(
                        f"Created CLOSE of {self.position.size} "
                        f"at {self.dataclose[0]} on day {dom}"
                    )
                    self.order = self.order_target_percent(target=0.0)
            if self.position.size < 0:
                if not 1 <= dom <= self.params.start_of_month:
                    print(
                        f"Created CLOSE of {self.position.size} "
                        f"at {self.dataclose[0]} on day {dom}"
                    )
                    self.order = self.order_target_percent(target=0.0)

The params tuple lets us adjust the day-of-month thresholds without editing the logic itself. The order_target_percent method tells Backtrader to size positions as a fraction of portfolio value, where -1 means fully short and 1 means fully long. Notice that notify_order resets self.order to None after each fill, which prevents the strategy from stacking multiple orders on consecutive bars. This is exactly the kind of guardrail that prevents subtle bugs in a backtest.

## Run the backtest and evaluate results

We load 20 years of TLT (long-term U.S. Treasury bond ETF) data, which gives us a long enough history to see how the strategy behaves across multiple market regimes.

In [ ]:
data = yf_data_to_bt_data(
    "TLT",
    start="2020-01-01",
    end="2024-12-31"
)

Cerebro is Backtrader's central engine. We configure it with our data feed, starting capital, strategy, a portfolio value observer for plotting, and two analyzers that capture return metrics we'll need for evaluation.

In [ ]:
cerebro = bt.Cerebro(stdstats=False)
cerebro.adddata(data)
cerebro.broker.setcash(1000.0)
cerebro.addstrategy(MonthlyFlows)
cerebro.addobserver(bt.observers.Value)
cerebro.addanalyzer(
    bt.analyzers.Returns, _name="returns"
)
cerebro.addanalyzer(
    bt.analyzers.TimeReturn, _name="time_return"
)
backtest_result = cerebro.run()

Setting stdstats=False disables Backtrader's default observers so we only track what we explicitly add. The TimeReturn analyzer records daily returns as a dictionary keyed by date, which we'll convert to a pandas DataFrame for QuantStats. Starting with $1,000 keeps the numbers easy to interpret while still producing realistic percentage-based results.

We extract the daily strategy returns from the backtest result and reshape them into a DataFrame that QuantStats can consume.

In [ ]:
returns_dict = (
    backtest_result[0].analyzers.time_return.get_analysis()
)
returns_df = (
    pd.DataFrame(
        list(returns_dict.items()),
        columns=["date", "return"]
    )
    .set_index("date")
)

Backtrader stores analyzers on the first element of the result list because cerebro.run() returns one result per strategy. Converting the dictionary to a DataFrame with a date index is the standard handoff format that QuantStats and most Python finance tools expect.

We load the same TLT price series as a benchmark, then generate a full QuantStats performance report comparing our strategy against a simple buy-and-hold approach.

In [ ]:
qs.reports.full(
    returns_df["return"],
    "SPY",
)

This is the most important step in the entire workflow. Comparing against a benchmark answers the question every backtest should ask: did the strategy actually add value, or would we have been better off just holding the asset? QuantStats produces dozens of metrics including Sharpe ratio, maximum drawdown, and win rate, giving us a professional-grade evaluation without writing any of that math ourselves.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.